# Análisis de parámetros: características de forma, estructura radial y textura
**PSIB 2026 Q2 — Grupo 9 (Del Carril, Escobar, Uthurralt)**

Continúa a `pipeline_2_segmentacion.ipynb` (etapas 1 a 4): en vez de volver a correr esas etapas, esta celda carga sus resultados ya guardados en `resultados/etapaN/<CONJUNTO>/` (`cargar_etapa2`, `cargar_etapa4`), así que primero hay que haber corrido ese notebook con el mismo `CONJUNTO`.
| Etapa | Qué hace | Archivo |
|---|---|---|
| 5 | Características y textura (GLCM) en borde vs. centro | `etapas/etapa5.py` |
| 6 | Decisión estadística: comparación entre clases y elección del umbral  | *pendiente* |
| 7 | Evaluación en testeo | *pendiente* |

In [6]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etapas.datos import listar_dataset
from etapas import e2_mejoramiento as e2
from etapas import e4_morfologia as e4

RUTA_DATASET = Path("Dataset_reducido")
CONJUNTO = "Entrenamiento"   # tiene que ser el mismo CONJUNTO con el que se corrió pipeline_2_segmentacion.ipynb
RUTA_RESULTADOS = Path("resultados")

df = listar_dataset(RUTA_DATASET, CONJUNTO)
print(f"Conjunto: {CONJUNTO} — {len(df)} imágenes")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Conjunto: Entrenamiento — 37 imágenes


### Carga de resultados guardados (etapas 2 y 4)

In [7]:
carpeta_e2 = RUTA_RESULTADOS / "etapa2" / CONJUNTO
resultados_e2 = {id_: e2.cargar_etapa2(carpeta_e2 / f"{id_}_e2.npz") for id_ in df.index}

carpeta_e4 = RUTA_RESULTADOS / "etapa4" / CONJUNTO
res_e4 = {id_: e4.cargar_etapa4(carpeta_e4 / f"{id_}_e4.npz") for id_ in df.index}

print(f"Cargadas etapa2 y etapa4 de {len(df)} imágenes desde {RUTA_RESULTADOS.resolve()}")

Cargadas etapa2 y etapa4 de 37 imágenes desde C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados


## 5. Características

Elegimos 6 parametros para analizar las heridas, parametros que tienen que ver con la forma, con la estructura y con las texturas.

### 5.1 Descriptores de forma

Funciones en `etapas/forma.py`, aplicadas sobre la lesión final (`res_e4[id_]["lesion"]`: la
máscara rellena recortada a la piel, la misma que usa la etapa 4 para definir zonas).

- **Circularidad** = 4·π·Área / Perímetro² del contorno externo. 1 = círculo perfecto; baja con
  anillos incompletos o lesiones alargadas.
- **Dimensión fractal del borde**: pendiente de log(N cajas) vs. log(1/tamaño de caja) sobre el
  contorno de 1 px (box-counting). Cerca de 1 = borde liso; más alto = borde festoneado — esperable
  en el anillo descamado de la TC.

Las lesiones anotadas como "difusa" (ocupan toda la foto) no tienen un borde real que medir:
`dim_fractal_borde` queda en NaN para esos casos, igual que no se genera banda de `borde` en la etapa 4.

In [8]:
from etapas import etapa5 as e5

descriptores = {id_: e5.descriptores_forma(res_e4[id_]["lesion"]) for id_ in df.index}
df_forma = pd.DataFrame.from_dict(descriptores, orient="index")
df_forma.index.name = "id"

carpeta_forma = RUTA_RESULTADOS / "forma" / CONJUNTO
ruta_forma = e5.guardar_csv(df_forma, carpeta_forma / "descriptores_forma.csv")
print(f"Guardados descriptores de forma de {len(df_forma)} imágenes en {ruta_forma.resolve()}")

Guardados descriptores de forma de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\forma\Entrenamiento\descriptores_forma.csv


### 5.2 Estructura radial: ratio de eritema y perfil de gradiente radial

Dos formas de medir lo mismo (si el eritema se concentra en el borde o está parejo), una como
número único y otra como curva. Usan `a_suave` (etapa 2) y las zonas `borde`/`centro`/`lesion`
de `res_e4` (etapa 4). Funciones en `etapas/forma.py`.

- **Ratio de eritema** $R_e = \bar{I}_{borde}(a^*) \,/\, \bar{I}_{centro}(a^*)$: $R_e > 1$ =
  borde más rojo que el centro (anillo); $R_e \approx 1$ = placa pareja.
- **Perfil de gradiente radial** $I(r)$: media de a* en bandas de distancia al centroide de la
  lesión, de r=0 (centroide) a r=1 (punto más alejado dentro de la lesión). Es la versión
  continua del mismo contraste: en una TC se espera que suba hacia r=1.

Mismo cuidado que con R_e: a* puede ser negativo, así que si la media del centro da ~0 el
cociente no tiene mucho sentido (queda en NaN); en ese caso conviene mirar el perfil completo.

In [9]:
df_radial = e5.procesar_dataset_radial(
    canales_a={id_: resultados_e2[id_]["a_suave"] for id_ in df.index},
    lesiones={id_: res_e4[id_]["lesion"] for id_ in df.index},
    bordes={id_: res_e4[id_]["borde"] for id_ in df.index},
    centros={id_: res_e4[id_]["centro"] for id_ in df.index},
    n_bins=10,
)

carpeta_radial = RUTA_RESULTADOS / "estructura_radial" / CONJUNTO
ruta_radial = e5.guardar_csv(df_radial, carpeta_radial / "estructura_radial.csv")
print(f"Guardados descriptores de estructura radial de {len(df_radial)} imágenes en {ruta_radial.resolve()}")


Guardados descriptores de estructura radial de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\estructura_radial\Entrenamiento\estructura_radial.csv


### 5.4 Textura: entropía y contraste GLCM (borde vs. centro)

Funciones en `etapas/textura.py`. Usan `L` **sin filtrar** de la etapa 2 (`resultados_e2[id_]["L"]`,
no `L_suave` -- la mediana borra justo las escamas/pápulas que la GLCM tiene que medir) dentro de
`borde` y `centro` (etapa 4). La GLCM se calcula por separado en cada zona, contando solo pares de
píxeles donde los dos extremos están dentro de la misma región (si no, un par borde-centro
contaminaría la textura de una zona con la de la otra).

$$ENT = -\sum_{i,j} P(i,j)\log_2 P(i,j) \qquad CON = \sum_{i,j} (i-j)^2 \, P(i,j)$$

Hipótesis de la tabla: en TC, $ENT_{borde} > ENT_{centro}$ (desorganización por escamas/pápulas
en el anillo) y $CON_{borde} \gg CON_{centro}$ (contraste local alto en esa franja). Abajo se
calculan `ent_diff` y `con_diff` (borde − centro) y se chequea esa hipótesis directamente: en
cuántas imágenes de cada clase el signo da el esperado.

In [10]:
df_textura = e5.procesar_dataset_textura(
    canales_L={id_: resultados_e2[id_]["L"] for id_ in df.index},
    bordes={id_: res_e4[id_]["borde"] for id_ in df.index},
    centros={id_: res_e4[id_]["centro"] for id_ in df.index},
)

carpeta_textura = RUTA_RESULTADOS / "textura" / CONJUNTO
ruta_textura = e5.guardar_csv(df_textura, carpeta_textura / "textura.csv")
print(f"Guardados descriptores de textura de {len(df_textura)} imágenes en {ruta_textura.resolve()}")

Guardados descriptores de textura de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\textura\Entrenamiento\textura.csv
